In [12]:
import pandas as pd
import re

In [13]:
#Now are going to load raw_company_data.
raw_company_data_source_path = '/Users/sundeepseethala/Downloads/raw_companies.csv'
raw_company_data = pd.read_csv(raw_company_data_source_path)
#raw_company_data.head()

In [14]:
# Now we are going to apply transformations on raw_compnay to split Address as Street, City, State, Zip.
df = pd.DataFrame(raw_company_data)
df

,Company_Name,Website,Address,Revenue_in_Millions,Industry
0,Potter LLC,maldonado.biz,"610 3rd St., Santa Rosa, California, 95404",407.81,Technology
1,Jackson Ltd,steele-barnett.com,"15045 River Rd., Guerneville, California, 95446",480.74,Healthcare
2,Torres Ltd,banks-carroll.com,"912 Cole Street, #338, San Francisco, Californ...",414.27,Hospitality
3,"Rich, Matthews and Jimenez",miller-burke.org,"1080 W. Old San Marcos Blvd., San Marcos, Cali...",370.96,Manufacturing
4,"Spears, Ellis and Rice",foster.net,"901 Gilman St., Berkeley, California, 94710",460.03,Retail
...,...,...,...,...,...
95,"Cook, Clayton and Suarez",moody.info,"146 Bodman Pl., Redbank, New Jersey, 7701",406.43,Hospitality
96,"Chaney, Vance and Bernard",warren-horne.com,"905 36th Place, Rio Rancho, New Mexico, 87124",411.18,Healthcare
97,Williams LLC,thompson-joseph.com,"106 Des Georges Lane, Taos, New Mexico, 87571",454.94,Finance
98,Young-Robinson,osborne.biz,"P.O. Box 154, Embudo, New Mexico, 87531",230.58,Finance


In [15]:
#Function to split address into street, city, state, zip
def parse_address(addr):
    street, city, state, zipcode = None, None, None, None
    parts = [p.strip() for p in addr.split(',') if p.strip()]

    # Check for ZIP code at the end
    zip_match = re.search(r'(\d{5})$', addr)
    if zip_match:
        zipcode = zip_match.group(1)

    # --- Handle small combinations first ---
    # Case 1: City + ZIP (no state)
    if len(parts) == 2 and re.fullmatch(r'\d{5}', parts[1]):
        city, zipcode = parts[0], parts[1]
        return pd.Series([street, city, state, zipcode])

    # Case 2: State + ZIP (no city)
    if len(parts) == 2 and not re.fullmatch(r'\d{5}', parts[0]) and re.fullmatch(r'\d{5}', parts[1]):
        state, zipcode = parts[0], parts[1]
        return pd.Series([street, city, state, zipcode])

    # Case 3: City + State (no zip)
    if len(parts) == 2 and not re.fullmatch(r'\d{5}', parts[1]):
        city, state = parts
        return pd.Series([street, city, state, zipcode])

    # --- Handle longer addresses ---
    # Full address: Street, City, State, Zip
    if len(parts) >= 4:
        street = ', '.join(parts[:-3])
        city, state, zipcode = parts[-3], parts[-2], parts[-1]
        return pd.Series([street, city, state, zipcode])

    # Case 4: City, State, Zip
    if len(parts) == 3:
        city, state, zipcode = parts
        return pd.Series([street, city, state, zipcode])

    # Fallback: single value (assume city)
    if len(parts) == 1:
        city = parts[0]
        return pd.Series([street, city, state, zipcode])

    return pd.Series([street, city, state, zipcode])

In [18]:
df[['Street', 'City', 'State', 'Zip']] = df['Address'].apply(parse_address)
#df.duplicated().sum()
df.head()



,Company_Name,Website,Address,Revenue_in_Millions,Industry,Street,City,State,Zip
0,Potter LLC,maldonado.biz,"610 3rd St., Santa Rosa, California, 95404",407.81,Technology,610 3rd St.,Santa Rosa,California,95404
1,Jackson Ltd,steele-barnett.com,"15045 River Rd., Guerneville, California, 95446",480.74,Healthcare,15045 River Rd.,Guerneville,California,95446
2,Torres Ltd,banks-carroll.com,"912 Cole Street, #338, San Francisco, Californ...",414.27,Hospitality,"912 Cole Street, #338",San Francisco,California,94117
3,"Rich, Matthews and Jimenez",miller-burke.org,"1080 W. Old San Marcos Blvd., San Marcos, Cali...",370.96,Manufacturing,1080 W. Old San Marcos Blvd.,San Marcos,California,92069
4,"Spears, Ellis and Rice",foster.net,"901 Gilman St., Berkeley, California, 94710",460.03,Retail,901 Gilman St.,Berkeley,California,94710
